# How Far Back Can a Vanilla RNN See?

**Topic:** Backpropagation through time &nbsp;·&nbsp; **Difficulty:** Medium

The gradient of an RNN loss with respect to an input `k` steps back is a product
of `k` Jacobians. Products of matrices whose spectral radius is not exactly 1 go
to zero or infinity geometrically, and "exactly 1" is not something gradient
descent maintains.

Measure the decay directly for three weight scales. The `tanh` derivative caps
each step at 1 and the units saturate, so even a spectral radius above 1 ends up
vanishing (it just takes longer). Read the effective memory off the curve.

## The data (created in the next cell)

- Hidden size 32, sequence length 80, `tanh` recurrence
  `h_t = tanh(W h_{t-1} + U x_t)`.
- Three spectral radii for `W`: `0.7`, `1.0`, `1.3`.
- Inputs `X` `(80, 8)` fixed across the three runs.

## Your task

With **NumPy only**:

1. For each spectral radius, scale a random `W` so its largest eigenvalue
   magnitude equals the target, run the recurrence forward for 80 steps, then
   backpropagate `dL/dh_80 = ones`, recording the norm at each backprop
   step, so the entry at index `t` holds `||dL/dh_{t+1}||` (index 79 holds
   `||dL/dh_80||`).
2. `effective_memory`: for each run, the largest lag `L` such that the gradient
   at `t = 80 - L` is at least `1e-6` times the gradient at `t = 80`.
3. Set `answers` to a dict with `"grad_norms"` `(3, 80)`, `"effective_memory"`
   `(3,)`, and `"decay_rate_07"`: the geometric mean of the consecutive ratios
   `grad_norms[0][t + 1] / grad_norms[0][t]` over its last 40 entries
   (39 ratios, each above 1, so the reciprocal is the per-step decay).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks all three runs decay backwards through time and that the effective memory
grows with the spectral radius.

In [ ]:
import numpy as np

rng = np.random.default_rng(4002)
H, T, D = 32, 80, 8
W_base = rng.normal(0, 1, (H, H))
W_base = W_base / np.max(np.abs(np.linalg.eigvals(W_base)))
U = rng.normal(0, 0.4, (H, D))
X = rng.normal(0, 1, (T, D))
radii = [0.7, 1.0, 1.3]
print(f"hidden {H}, sequence {T}, spectral radii {radii}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("grad_norms", [3, 80], 57.63543825799367, 36.747766508968795, 1e-06),
    ("effective_memory", [3], 79.0, -60.30575463094793, 1e-06),
    ("decay_rate_07", [], 2.47762976711631, -0.4148494096512648, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "grad_norms" in answers:
    _g = np.asarray(answers["grad_norms"], dtype=float)
    _e = np.asarray(answers["effective_memory"], dtype=float)
    _all_decay = bool(np.all(_g[:, 0] < _g[:, -1]))
    _mono = bool(_e[0] < _e[1] < _e[2])
    print(f"  {'all three vanish':<24} {'ok' if _all_decay else 'wrong'}  "
          f"(t=0 norms {[f'{v:.1e}' for v in _g[:, 0]]})")
    print(f"  {'memory grows with radius':<24} {'ok' if _mono else 'wrong'}  "
          f"(lags {_e.astype(int).tolist()})")
    _ok = _ok and _all_decay and _mono

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Each backward step multiplies by `W^T diag(1 - h^2)`. The `tanh` derivative is at
most 1 and is far below it once units saturate, so the *effective* per-step factor
sits well under the spectral radius, which is why even the radius-1.3 run
vanishes instead of exploding. Larger radii buy more memory (15, 23, then 41
usable steps here) but never enough: at 80 steps every configuration has lost the
signal. This is the concrete statement behind "RNNs cannot model long-range
dependencies", and it is what gating was invented to fix.

```python
import numpy as np

def run(radius):
    W = W_base * radius
    h = np.zeros(H)
    hs = []
    for t in range(T):
        h = np.tanh(W @ h + U @ X[t])
        hs.append(h.copy())
    g = np.ones(H)
    norms = np.zeros(T)
    for t in reversed(range(T)):
        norms[t] = float(np.linalg.norm(g))
        g = W.T @ (g * (1 - hs[t] ** 2))
    return norms

grad_norms = np.stack([run(r) for r in radii])

eff = []
for row in grad_norms:
    ref = row[-1]
    ok = np.flatnonzero(row >= 1e-6 * ref)
    eff.append(float(T - 1 - ok.min()) if len(ok) else 0.0)
eff = np.array(eff)

r07 = grad_norms[0]
ratios = r07[-40:][1:] / np.maximum(r07[-40:][:-1], 1e-300)
decay = float(np.exp(np.mean(np.log(np.maximum(ratios, 1e-300)))))

answers = {
    "grad_norms": grad_norms,
    "effective_memory": eff,
    "decay_rate_07": decay,
}
```